# Data Processing

Processes raw files and creates a nice csv file

In [1]:
import pandas as pd
import numpy as np
from os import listdir
from pathlib import Path

from agripharm_track3_ethnohack_2026.forecasting.utils import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
RAW_DATA_DIR

WindowsPath('C:/Users/chris/Desktop/MyCode/ethnohack_2026/AgriPharm/AgriPharm_Track3_EthnoHack_2026/data/raw')

In [3]:
raw_filenames = [RAW_DATA_DIR / Path(f) for f in listdir(RAW_DATA_DIR) if Path(f).suffix == ".xlsx"]

In [4]:
df = pd.read_excel(raw_filenames[0], sheet_name=9, skiprows=5)
df

,Code ATC5,Libellé ATC5,Base de remboursement 2021-01,Nombre de boites remboursées \n2021-01,Montant remboursé \n2021-01,Base de remboursement 2021-02,Nombre de boites remboursées \n2021-02,Montant remboursé \n2021-02,Base de remboursement 2021-03,Nombre de boites remboursées \n2021-03,Montant remboursé \n2021-03,Base de remboursement 2021-04,Nombre de boites remboursées \n2021-04,Montant remboursé \n2021-04,Base de remboursement 2021-05,Nombre de boites remboursées \n2021-05,Montant remboursé \n2021-05,Base de remboursement 2021-06,Nombre de boites remboursées \n2021-06,Montant remboursé \n2021-06
0,A01AA01,SODIUM FLUORURE,6.233800e+03,3732,2.068914e+03,6.263680e+03,3781,2.095821e+03,7.138570e+03,4268,2.375731e+03,6.247790e+03,3706,2.088466e+03,5.482440e+03,3215,1.831829e+03,6.147300e+03,3637,2.093666e+03
1,A01AA02,SODIUM MONOFLUOROPHOSPHATE,5.546068e+04,5164,4.269829e+04,5.417906e+04,5071,4.114861e+04,6.317682e+04,5918,4.774601e+04,5.795225e+04,5429,4.423414e+04,5.527642e+04,5181,4.188342e+04,6.156594e+04,5771,4.651061e+04
2,A01AA51,SODIUM FLUORURE EN ASSOCIATION,7.751732e+04,29201,2.370117e+04,7.692961e+04,28948,2.343427e+04,8.436429e+04,31751,2.583929e+04,7.297094e+04,27456,2.229768e+04,6.320032e+04,23755,1.927660e+04,7.181397e+04,27009,2.198152e+04
3,A01AB03,CHLORHEXIDINE,1.439890e+06,561367,2.893059e+05,1.637991e+06,622723,3.229237e+05,1.928634e+06,725890,3.817712e+05,1.777838e+06,669520,3.510056e+05,1.534571e+06,575851,3.061034e+05,1.873613e+06,702182,3.700958e+05
4,A01AB09,MICONAZOLE,2.955899e+05,43539,2.140734e+05,2.970149e+05,43828,2.146263e+05,3.508785e+05,51445,2.543139e+05,3.189760e+05,46867,2.319869e+05,2.805541e+05,41069,2.038459e+05,3.379457e+05,49693,2.455082e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1185,V08CA08,GADOBENIQUE ACIDE,1.240735e+05,2589,9.210719e+04,1.185995e+05,2468,8.739619e+04,1.397384e+05,2960,1.020834e+05,1.312110e+05,2775,9.615663e+04,1.168729e+05,2475,8.582647e+04,1.388920e+05,2950,1.019573e+05
1186,V08CA09,GADOBUTROL,2.547547e+06,36013,1.817326e+06,2.519077e+06,35844,1.789359e+06,2.870168e+06,40815,2.036758e+06,2.619485e+06,37056,1.855198e+06,2.375070e+06,33612,1.686153e+06,2.809371e+06,39578,1.993789e+06
1187,V08DA05,SOUFRE HEXAFLUORURE,1.033440e+05,1561,8.011790e+04,9.057303e+04,1374,7.146701e+04,1.041350e+05,1580,8.082727e+04,8.842194e+04,1342,6.859481e+04,8.317516e+04,1262,6.456878e+04,9.816779e+04,1491,7.593679e+04
1188,W,Homéopathie,3.892307e+06,1725254,4.631871e+05,2.192165e+06,385884,4.656791e+04,2.336605e+06,310490,1.510890e+04,2.120927e+06,259530,7.499013e+03,1.848019e+06,217293,5.429830e+03,2.185423e+06,254509,4.868405e+03


In [5]:
INDEX_COLS = ["ATC5", "full_name"]

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # basic cleaning
    df.columns = df.columns.str.replace('\n', '', regex=False)
    df.columns = df.columns.str.strip().str.lower()
    df = df.dropna(how='all')

    # numeric restoration
    numeric_cols = df.columns[2:]
    for col in numeric_cols:
        # 'coerce' converts non numeric values to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # use only number of boxes
    columns_to_drop = [col for col in numeric_cols if not col.startswith("nombre")]
    df = df.drop(columns=columns_to_drop)
    df.columns = INDEX_COLS + [
        col.replace("nombre de boites remboursées", "").strip() 
        for col in df.columns[2:]
    ]

    # drop the last 2 rows that contain useless information for the project
    df = df.iloc[:-2, :]
    return df

In [6]:
clean_df(df)

,ATC5,full_name,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06
0,A01AA01,SODIUM FLUORURE,3732,3781,4268,3706,3215,3637
1,A01AA02,SODIUM MONOFLUOROPHOSPHATE,5164,5071,5918,5429,5181,5771
2,A01AA51,SODIUM FLUORURE EN ASSOCIATION,29201,28948,31751,27456,23755,27009
3,A01AB03,CHLORHEXIDINE,561367,622723,725890,669520,575851,702182
4,A01AB09,MICONAZOLE,43539,43828,51445,46867,41069,49693
...,...,...,...,...,...,...,...,...
1183,V08CA03,GADODIAMIDE,2,1,0,0,0,0
1184,V08CA04,GADOTERIDOL,18351,18214,21472,20165,17739,20292
1185,V08CA08,GADOBENIQUE ACIDE,2589,2468,2960,2775,2475,2950
1186,V08CA09,GADOBUTROL,36013,35844,40815,37056,33612,39578


In [7]:
dfs = []

for f in raw_filenames:
    cur_df = pd.read_excel(f, sheet_name=9, skiprows=5)
    cur_df = clean_df(cur_df)
    dfs.append(cur_df.set_index(INDEX_COLS))

total_df = pd.concat(dfs, axis=1)
total_df = total_df.sort_index(axis=1)
total_df = total_df.astype("Int64")
total_df.reset_index()
total_df

,,2012-01,2012-02,2012-03,2012-04,2012-05,2012-06,2012-07,2012-08,2012-09,2012-10,...,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07
ATC5,full_name,,,,,,,,,,,,,,,,,,,,,
A01AA01,SODIUM FLUORURE,19088,18432,18333,17481,17231,18601,16921,13451,19842,22699,...,1189,1078,1140,1064,1016,1219,1121,997,1178,1101
A01AA02,SODIUM MONOFLUOROPHOSPHATE,3361,3257,3550,3492,3506,3570,3451,3216,3328,3884,...,6268,5388,6244,5449,5509,6049,5696,5211,6370,6085
A01AA51,SODIUM FLUORURE EN ASSOCIATION,154831,146673,147993,140019,141000,147924,140096,114336,144426,161754,...,9983,8431,10011,8634,8507,9087,7964,7145,8080,7188
A01AB03,CHLORHEXIDINE,400945,406123,431731,409432,407284,445268,441589,326885,393664,484590,...,836319,723897,819258,718972,775633,840313,713712,673703,839906,783903
A01AB09,MICONAZOLE,91278,82751,90123,80547,82156,89623,92825,79044,75777,94966,...,1030,875,1017,822,907,1067,862,864,966,996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
L01XE39,MIDOSTAURINE,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
L01XE44,LORLATINIB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
L01XE53,LAROTRECTINIB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
total_df.to_csv(Path(PROCESSED_DATA_DIR / "atc5_drug_history.csv"))

### ATC5 selection

Initially:
- M04AC01
- C01AA05
- S01EB01
- A04AD01
- S01FA01

Maybe later:
- P01BF01
- L01CA04
- L01CB01

In [9]:
SELECTED_ATC5 = ["M04AC01", "C01AA05", "S01EB01", "A04AD01", "S01FA01",]
selected_df = total_df.loc[SELECTED_ATC5, :]
selected_df

,,2012-01,2012-02,2012-03,2012-04,2012-05,2012-06,2012-07,2012-08,2012-09,2012-10,...,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07
ATC5,full_name,,,,,,,,,,,,,,,,,,,,,
M04AC01,COLCHICINE,190768,174553,196767,186323,200524,201824,205607,189731,171909,199195,...,108279,92044,115023,103432,104911,120947,113123,104976,133152,138606
C01AA05,DIGOXINE,236125,217493,230538,220449,234628,223330,232447,223257,200310,232858,...,52344,45836,53088,47946,45547,50798,47271,44334,50094,48464
S01EB01,PILOCARPINE,14681,13736,15573,15081,16332,16070,16621,14882,14436,16295,...,11206,10140,10682,10755,10438,11693,10208,9561,11440,9653
A04AD01,SCOPOLAMINE,1097,1114,1287,977,1237,1127,1127,1083,972,1422,...,2704,2407,2816,2720,2345,2477,2350,1975,2165,2293
S01FA01,ATROPINE,19757,19250,20112,19278,18950,19741,18771,15241,17238,20687,...,21280,17199,19969,18336,19198,20972,19456,17155,20588,18743


In [10]:
print("number of na values per row:\n")
print(selected_df.isna().sum(axis=1))

number of na values per row:

ATC5     full_name  
M04AC01  COLCHICINE     0
C01AA05  DIGOXINE       0
S01EB01  PILOCARPINE    0
A04AD01  SCOPOLAMINE    0
S01FA01  ATROPINE       0
dtype: int64


In [11]:
selected_df.to_csv(Path(PROCESSED_DATA_DIR / "selected_atc5_drug_history.csv"))

In [12]:
train_df = selected_df.copy().reset_index()
cols = train_df.columns
train_df = train_df.set_index(cols[0]).drop(columns=[cols[1]]).T
train_df.index = pd.to_datetime(train_df.index, format="%Y-%m")
train_df = train_df.asfreq("MS") # montlhy frequency (Monthly Start)
train_df

ATC5,M04AC01,C01AA05,S01EB01,A04AD01,S01FA01
2012-01-01,190768,236125,14681,1097,19757
2012-02-01,174553,217493,13736,1114,19250
2012-03-01,196767,230538,15573,1287,20112
2012-04-01,186323,220449,15081,977,19278
2012-05-01,200524,234628,16332,1237,18950
...,...,...,...,...,...
2026-03-01,120947,50798,11693,2477,20972
2026-04-01,113123,47271,10208,2350,19456
2026-05-01,104976,44334,9561,1975,17155
2026-06-01,133152,50094,11440,2165,20588


In [13]:
train_df.to_csv(Path(PROCESSED_DATA_DIR / "initial_train_data.csv"))